The goal of this notebook is to explore the implementation of boundary conditions in NGSolve with respect to the Prolongation matrix, the inverse function, and the Projector function.

In [143]:
# import necessary packages
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
import numpy as np
import time

In [144]:
coarse_mesh = Mesh(unit_square.GenerateMesh(maxh=1/16))
fesc = H1(coarse_mesh, order=1, dirichlet="left|right")
fine_mesh = Mesh(coarse_mesh.ngmesh.Copy())
fesf = H1(fine_mesh, order=1, dirichlet="left|right")
fesf.mesh.Refine()
P = fesf.Prolongation().CreateMatrix(fesf.mesh.levels-1)
PT = P.CreateTranspose()


In [145]:
coarse_mesh2 = Mesh(unit_square.GenerateMesh(maxh=1/16))
fesc2 = H1(coarse_mesh2, order=1)
fine_mesh2 = Mesh(coarse_mesh2.ngmesh.Copy())
fesf2 = H1(fine_mesh2, order=1)
fesf2.mesh.Refine()
P2 = fesf2.Prolongation().CreateMatrix(fesf2.mesh.levels-1)
PT2 = P2.CreateTranspose()
#print("free dofs of fesc2 without \"dirichlet\" flag:\n",fesc2.FreeDofs())
#print("free dofs of fesf2 without \"dirichlet\" flag:\n", fesf2.FreeDofs())
#print("dofs of fesf2fine:\n",fesf2.GetNDofLevel(1))
#print("dofs of fesf2 lower level (coarse):\n",fesf2.GetNDofLevel(0))

In [137]:
help(Draw)

Help on function Draw in module netgen.webgui:

Draw(obj, *args, show=True, **kwargs)
    Visualise a mesh or field in the webgui (Jupyter or standalone).

    The object type determines how it is rendered.  Netgen meshes
    (``netgen.meshing.Mesh``) are supported out of the box.  When
    *ngsolve* is imported, ``ngsolve.Mesh``, ``CoefficientFunction``
    and ``GridFunction`` are registered as well via
    :func:`register_draw_type`.

    Parameters
    ----------
    obj : mesh, function, or any registered draw type
        The object to visualise.  For an ``ngsolve.CoefficientFunction``
        pass the mesh or region as the first positional argument.
    show : bool
        Display the widget immediately in Jupyter (default ``True``).
    order : int
        Polynomial order used for visualisation (default 2).
    draw_vol : bool
        Draw volume elements (default ``True``).
    draw_surf : bool
        Draw surface elements (default ``True``).
    deformation : bool or GridFu

In [175]:
u, v = fesf.TnT()
a = BilinearForm(grad(u)*grad(v)*dx).Assemble()
f = LinearForm(v*dx).Assemble()
a_fine = a.GetMatrixLevel()
m = a_fine.CreateSmoother(fesf.FreeDofs(), GS=True)
x0 = CoefficientFunction(sin(pi*x)*sin(pi*y)+(1/10)*sin(10*pi*x)*sin(10*pi*y))
xi = GridFunction(fesf)
xi.Set(x0)
yi = GridFunction(fesf)
s = Draw(xi,fine_mesh,settings={"camera": {"transformations": [{"type": "rotateX", "angle": -60}]},"deformation":0.5})
smootha = m @ a_fine
for i in range(10):
    smootha.Mult(xi.vec, yi.vec.data)
    xi.vec.data -= yi.vec
    time.sleep(1)
    s.Redraw()
yi.vec.data = a.mat * xi.vec

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'camera': {'transformations'…

In [210]:
phi, nu = fesc.TnT()
a_coarse = BilinearForm(grad(phi)*grad(nu)*dx).Assemble()
acinv = a_coarse.mat.Inverse(freedofs=fesc.FreeDofs(True))
print("Checking A coarse\n")
#print(a_coarse.mat)
alt_coarse = PT @ a_fine @ P
#print(alt_coarse)
#ac_diff = a_coarse.mat - alt_coarse
#print(ac_diff)


Checking A coarse



In [211]:
yc = GridFunction(fesc)
yc.vec.data = PT * yi.vec # this supposes that the result of the GS smoothing is a vector yi of the "fine" length
Draw(yc,coarse_mesh, settings={"camera": {"transformations": [{"type": "rotateX", "angle": -60}]},"deformation":0.5})


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'camera': {'transformations'…

BaseWebGuiScene

In [212]:
xc = GridFunction(fesc)
#print(xc.vec)
xc.vec.data = acinv * yc.vec
#print(xc.vec)
Draw(xc,coarse_mesh, settings={"camera": {"transformations": [{"type": "rotateX", "angle": -60}]},"deformation":0.5})

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'camera': {'transformations'…

BaseWebGuiScene

In [163]:
yf = GridFunction(fesf)
yf.vec.data = P * yi.vec